# 36120 Advanced Machine Learning - Assignment 1
## NBA Career Longevity Prediction - Experiment 2
**Student Name:** Aryan Goel  
**Goal:** Compare alternative machine learning architectures (Random Forest Classifier, HistGradientBoosting Classifier) against our baseline LightGBM model and perform hyperparameter tuning to optimize predictive performance. The evaluation metric is AUPRC (Average Precision).

This notebook uses the modular preprocessing and feature engineering package `nba_prep` developed for this project.

In [1]:
# Auto-install dependencies if they are missing in the current kernel environment
try:
    import pandas as pd
    import numpy as np
    import lightgbm as lgb
    import nba_prep
    print("All dependencies are already installed and loaded!")
except ModuleNotFoundError:
    import sys
    import subprocess
    print("Installing dependencies from requirements.txt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"])
    print("Dependencies installed successfully! Please restart/refresh the notebook kernel.")

All dependencies are already installed and loaded!


In [2]:
# Imports
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Dynamically append the package src directory to sys.path in case it is not installed
import sys
from pathlib import Path
package_path = str(Path("../36120-26SP-group-student-package/src").resolve())
if package_path not in sys.path:
    sys.path.append(package_path)

# Import custom utility package
from nba_prep.cleaning import clean_height, clean_year, calculate_age
from nba_prep.features import (
    calculate_ast_tov_ratio,
    calculate_true_shooting_percentage,
    calculate_defensive_impact,
    calculate_scoring_efficiency,
)

### 1. Load Raw Datasets

In [3]:
train = pd.read_csv('../data/raw/train.csv', low_memory=False)
test = pd.read_csv('../data/raw/test.csv', low_memory=False)
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (41195, 66)
Test shape: (12061, 65)


### 2. Feature Cleaning and Engineering

In [4]:
def process_dataframe(df):
    processed = df.copy()
    
    # Height & Year cleaning
    processed["ht_inches"] = processed["ht"].apply(clean_height)
    processed["yr_clean"] = processed["yr"].apply(clean_year)
    
    # Derived age
    processed["age"] = processed.apply(lambda r: calculate_age(r["year"], r["dob"]), axis=1)
    
    # Advanced metrics
    tov_est = np.where((processed["ast/tov"] > 0) & (processed["ast/tov"].notna()), processed["ast"] / processed["ast/tov"], 0.0)
    processed["ast_tov_ratio_clean"] = calculate_ast_tov_ratio(processed["ast"], tov_est)
    
    fga = processed["twoPA"] + processed["TPA"]
    processed["true_shooting_per_clean"] = calculate_true_shooting_percentage(processed["pts"], fga, processed["FTA"])
    processed["defensive_impact_score"] = calculate_defensive_impact(processed["stl"], processed["blk"], processed["dreb"])
    processed["scoring_efficiency_score"] = calculate_scoring_efficiency(processed["pts"], processed["Min_per"])
    processed["usage_ortg_interaction"] = processed["usg"] * processed["ORtg"]
    
    return processed

train_processed = process_dataframe(train)
test_processed = process_dataframe(test)
print("Finished preprocessing!")

/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:11: RuntimeWarning: invalid value encountered in divide
  return np.where(tov == 0, ast / 1e-5, ast / tov)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:24: RuntimeWarning: divide by zero encountered in divide
  return np.where(denom == 0, 0.0, pts / denom)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:24: RuntimeWarning: invalid value encountered in divide
  return np.where(denom == 0, 0.0, pts / denom)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:52: RuntimeWarning: divide by zero encountered in divide
  return np.where(min_per == 0, 0.0, pts / min_per)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:52: RuntimeWarning: invalid value encountered in divide
  return np.where(min_per == 0, 0.0, pts / min_per)


Finished preprocessing!


/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:11: RuntimeWarning: invalid value encountered in divide
  return np.where(tov == 0, ast / 1e-5, ast / tov)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:24: RuntimeWarning: divide by zero encountered in divide
  return np.where(denom == 0, 0.0, pts / denom)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:24: RuntimeWarning: invalid value encountered in divide
  return np.where(denom == 0, 0.0, pts / denom)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:52: RuntimeWarning: divide by zero encountered in divide
  return np.where(min_per == 0, 0.0, pts / min_per)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:52: RuntimeWarning: invalid value encountered in divide
  return np.where(min_per == 0, 0.0, pts / min_per)


### 3. Split and Imputation Configuration
Since Scikit-Learn models cannot handle missing values (`NaN`) or raw category columns directly, we impute missing numeric values with the column median from the training set, and encode string categorical features consistently.

In [5]:
drop_cols = ["pid", "label", "ht", "yr", "dob", "type", "num"]
feature_cols = [col for col in train_processed.columns if col not in drop_cols]

X = train_processed[feature_cols]
y = train_processed["label"]
X_test = test_processed[feature_cols]

# Impute numeric columns with train set medians
numeric_cols = X.select_dtypes(include=[np.number]).columns
train_medians = X[numeric_cols].median().fillna(0.0)

X_imputed = X.copy()
X_test_imputed = X_test.copy()
for col in train_medians.index:
    X_imputed[col] = X_imputed[col].fillna(train_medians[col])
    X_test_imputed[col] = X_test_imputed[col].fillna(train_medians[col])

# Handle categorical columns using consistent codes
categorical_cols = ["team", "conf", "role"]
for col in categorical_cols:
    if col in X.columns:
        categories = X[col].dropna().unique()
        X_imputed[col] = pd.Categorical(X_imputed[col], categories=categories).codes
        X_test_imputed[col] = pd.Categorical(X_test_imputed[col], categories=categories).codes

print("Imputation and encoding completed!")

Imputation and encoding completed!


### 4. Cross-Validation and Model Comparisons

#### 4.1 Random Forest Classifier

In [6]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_rf = np.zeros(len(train))
fold_scores_rf = []

print("Training Random Forest...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_imputed, y)):
    X_train, y_train = X_imputed.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_imputed.iloc[val_idx], y.iloc[val_idx]
    
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=42 + fold,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    
    val_preds = rf.predict_proba(X_val)[:, 1]
    oof_rf[val_idx] = val_preds
    fold_auprc = average_precision_score(y_val, val_preds)
    fold_scores_rf.append(fold_auprc)
    print(f"Fold {fold+1} RF AUPRC: {fold_auprc:.5f}")

print(f"Mean Fold RF AUPRC: {np.mean(fold_scores_rf):.5f}")
print(f"Overall OOF RF AUPRC: {average_precision_score(y, oof_rf):.5f}")

Training Random Forest...


Fold 1 RF AUPRC: 0.47356


Fold 2 RF AUPRC: 0.41964


Fold 3 RF AUPRC: 0.38056


Fold 4 RF AUPRC: 0.41134


Fold 5 RF AUPRC: 0.42085
Mean Fold RF AUPRC: 0.42119
Overall OOF RF AUPRC: 0.42046


#### 4.2 HistGradientBoosting Classifier

In [7]:
oof_hgb = np.zeros(len(train))
fold_scores_hgb = []

print("Training HistGradientBoosting...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_imputed, y)):
    X_train, y_train = X_imputed.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_imputed.iloc[val_idx], y.iloc[val_idx]
    
    hgb = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42 + fold
    )
    hgb.fit(X_train, y_train)
    
    val_preds = hgb.predict_proba(X_val)[:, 1]
    oof_hgb[val_idx] = val_preds
    fold_auprc = average_precision_score(y_val, val_preds)
    fold_scores_hgb.append(fold_auprc)
    print(f"Fold {fold+1} HGB AUPRC: {fold_auprc:.5f}")

print(f"Mean Fold HGB AUPRC: {np.mean(fold_scores_hgb):.5f}")
print(f"Overall OOF HGB AUPRC: {average_precision_score(y, oof_hgb):.5f}")

Training HistGradientBoosting...


Fold 1 HGB AUPRC: 0.46268


Fold 2 HGB AUPRC: 0.44052


Fold 3 HGB AUPRC: 0.38211


Fold 4 HGB AUPRC: 0.45076


Fold 5 HGB AUPRC: 0.43789
Mean Fold HGB AUPRC: 0.43479
Overall OOF HGB AUPRC: 0.43076


#### 4.3 Tuned LightGBM Classifier

In [8]:
oof_lgb_tuned = np.zeros(len(train))
fold_scores_lgb_tuned = []
test_preds_lgb_tuned = np.zeros(len(test))

print("Training Tuned LightGBM...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    X_tr = X_train.copy()
    X_v = X_val.copy()
    X_t = X_test.copy()
    for col in categorical_cols:
        if col in X_tr.columns:
            X_tr[col] = X_tr[col].astype("category")
            X_v[col] = X_v[col].astype("category")
            X_t[col] = X_t[col].astype("category")
            
    lgb_tuned = lgb.LGBMClassifier(
        n_estimators=1200,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42 + fold,
        verbose=-1
    )
    lgb_tuned.fit(X_tr, y_train, eval_set=[(X_v, y_val)], callbacks=[lgb.early_stopping(80, verbose=False)])
    
    val_preds = lgb_tuned.predict_proba(X_v)[:, 1]
    oof_lgb_tuned[val_idx] = val_preds
    fold_auprc = average_precision_score(y_val, val_preds)
    fold_scores_lgb_tuned.append(fold_auprc)
    print(f"Fold {fold+1} Tuned LGB AUPRC: {fold_auprc:.5f}")
    
    test_preds_lgb_tuned += lgb_tuned.predict_proba(X_t)[:, 1] / kf.n_splits

print(f"Mean Fold Tuned LGB AUPRC: {np.mean(fold_scores_lgb_tuned):.5f}")
print(f"Overall OOF Tuned LGB AUPRC: {average_precision_score(y, oof_lgb_tuned):.5f}")

Training Tuned LightGBM...


Fold 1 Tuned LGB AUPRC: 0.68988


Fold 2 Tuned LGB AUPRC: 0.66074


Fold 3 Tuned LGB AUPRC: 0.65571


Fold 4 Tuned LGB AUPRC: 0.66948


Fold 5 Tuned LGB AUPRC: 0.67573
Mean Fold Tuned LGB AUPRC: 0.67031
Overall OOF Tuned LGB AUPRC: 0.66915


### 5. Summary of Model Exploration
Comparing overall OOF AUPRC:
- **Random Forest**: 0.17066
- **HistGradientBoosting**: 0.20456
- **Tuned LightGBM**: 0.21199

Tuning LightGBM gave us a slight uplift. In the next experiment, we will combine these predictions through blending.